<a href="https://colab.research.google.com/github/CN-LEON-DX/AI/blob/main/TRAIN_Model_CFM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# dataset preparing
!pip install -q datasets torchcfm
#dataset
from datasets import load_dataset

ds = load_dataset("wanhin/naruto-captions", split="train")

#Text Encoder
import torch
from sentence_transformers import SentenceTransformer

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")

text_encoder = SentenceTransformer("all-mpnet-base-v2").to(device)



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


transform = transforms.Compose([
    transforms.Resize((64, 64)),  # size 64x64
    transforms.ToTensor()         # convert to tensor and range [0, 1]
])

In [9]:
class CFMDataset(Dataset):
    def __init__(self, dataset, transform, text_encoder, device):
        self.dataset = dataset
        self.transform = transform
        self.text_encoder = text_encoder
        self.device = device
        self.images = dataset["image"]
        self.captions = dataset["text"]
        # Encode caption to vector embedding
        self.embed_captions = text_encoder.encode(
            list(self.captions), convert_to_tensor=True, device=self.device # must in list type
        )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Get and process iamge
        image = self.images[idx]
        image = self.transform(image)

        # Get caption và embedding each image
        caption = self.captions[idx]
        caption_embedding = self.embed_captions[idx]

        return {
            "image": image,
            "caption": caption,
            "caption_embedding": caption_embedding,
        }

In [10]:
#  dataset
train_ds = CFMDataset(ds, transform, text_encoder, device)

# dataloader
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

In [11]:
import math
import torch
import torch.nn as nn
from torchcfm.models.unet import UNetModel

def timestep_embedding(timesteps, dim, max_period=10000):
    """Create sinusoidal timestep embeddings.

    Args:
        timesteps: a 1-D Tensor of N indices, one per batch element. These may be fractional.
        dim: the dimension of the output.
        max_period: controls the minimum frequency of the embeddings.

    Returns:
        an [N x dim] Tensor of positional embeddings.
    """
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period)
        * torch.arange(start=0, end=half, dtype=torch.float32, device=timesteps.device)
        / half
    )
    args = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding


class UNetModelWithTextEmbedding(UNetModel):
    """UNet model that incorporates text embeddings for conditional generation."""

    def __init__(self, dim, num_channels, num_res_blocks, embedding_dim, *args, **kwargs):
        """Initialize the text-conditional UNet model.

        Args:
            dim: The dimension of the input data (1D, 2D, or 3D)
            num_channels: Base channel count for the model
            num_res_blocks: Number of residual blocks per downsample
            embedding_dim: Dimension of the text embeddings
            *args, **kwargs: Additional arguments for the base UNetModel
        """
        super().__init__(dim, num_channels, num_res_blocks, *args, **kwargs)

        # Linear layer to project text embeddings to the right dimension
        self.embedding_layer = nn.Linear(embedding_dim, num_channels * 4)

        # Linear layer to combine time and text embeddings
        self.fc = nn.Linear(num_channels * 8, num_channels * 4)

    def forward(self, t, x, text_embeddings=None):
        """Apply the model to an input batch, incorporating text embeddings.

        Args:
            t: Timestep values (batch_size,) or (batch_size, 1)
            x: Input tensor of shape (batch_size, channels, [dimensions])
            text_embeddings: Optional text conditioning embeddings

        Returns:
            Output tensor with same shape as input x
        """
        # Ensure timesteps is a 1D tensor
        timesteps = t
        while timesteps.dim() > 1:
            timesteps = timesteps[:, 0]

        # If scalar, repeat for each element in the batch
        if timesteps.dim() == 0:
            timesteps = timesteps.repeat(x.shape[0])

        # Store skip connections
        hs = []

        # Get time embeddings
        emb = self.time_embed(timestep_embedding(timesteps, self.model_channels))

        # Incorporate text embeddings if provided
        if text_embeddings is not None:
            text_embedded = self.embedding_layer(text_embeddings)
            emb = torch.cat([emb, text_embedded], dim=1)  # Concatenate along feature dim
            emb = self.fc(emb)  # Combine time and text information

        # Process through UNet
        h = x.type(self.dtype)

        # Downsampling path
        for module in self.input_blocks:
            h = module(h, emb)
            hs.append(h)  # Store for skip connection

        # Middle blocks
        h = self.middle_block(h, emb)

        # Upsampling path with skip connections
        for module in self.output_blocks:
            h = torch.cat([h, hs.pop()], dim=1)  # Apply skip connection
            h = module(h, emb)

        # Final projection
        h = h.type(x.dtype)
        return self.out(h)

In [ ]:
import torch
from tqdm import tqdm

# Khởi tạo model với các tham số phù hợp
model = UNetModelWithTextEmbedding(
    dim=(3, 64, 64),        # 3 kênh màu, kích thước ảnh 64x64
    num_channels=32,        # Số kênh cơ bản cho model
    num_res_blocks=1,       # Số lượng residual blocks
    embedding_dim=768       # Kích thước của text embedding
).to(device)

# Khởi tạo optimizer
optimizer = torch.optim.Adam(model.parameters())

# Số epoch huấn luyện
n_epochs = 20000

# Vòng lặp huấn luyện
for epoch in tqdm(range(n_epochs)):
    losses = []
    for batch in train_loader:
        optimizer.zero_grad()
        x1 = batch["image"].to(device)
        text_embeddings = batch["caption_embedding"].to(device)
        x0 = torch.randn_like(x1).to(device)
        t = torch.rand(x0.shape[0], 1, 1, 1).to(device)
        xt = t * x1 + (1 - t) * x0
        ut = x1 - x0
        t = t.squeeze()
        vt = model(t, xt, text_embeddings=text_embeddings)
        loss = torch.mean(((vt - ut) ** 2))
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    avg_loss = sum(losses) / len(losses)
    if (epoch + 1) % 500 == 0:
        print(f"Epoch [{epoch+1}/{n_epochs}], Loss: {avg_loss:.4f}")

torch.save(model.state_dict(), "cfm_model.pt")

  0%|          | 7/20000 [05:13<245:40:56, 44.24s/it]